> see `evaluate_properties_extraction.py` for most recent code.

# Evaluate Property Extraction

> y (ground truth) vs y_hat (LLM)

In [1]:
import os
import sys

import numpy as np
import pandas as pd

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# print(notebook_dir)
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

## Load Data

temperature = 0 for more control

In [2]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
# data_path = os.path.join(base_data_path, 'extract_prediction_properties/future-chronicle2050_fpb_synthethic-v1/openai/gpt-oss-120b/results.csv')
data_path = os.path.join(base_data_path, 'extract_prediction_properties-v1/evaluation/synthetic-fpb-chronicle2050-yt-news-timebank-mf_climate/openai_gpt-oss-120b/extracted_properties.csv')
df = DataProcessing.load_from_file(data_path)
df

,Input_Index,Sentence,Raw Response,Model Name,No Property,Source,Target,Date,Outcome,Dataset Source
0,0,"As the yen soared in recent years, Sansui's de...","{0: [""As"", ""the"", ""yen"", ""surged"", ""in"", ""rece...",openai/gpt-oss-120b,As|the|yen|surged|in|recent|years|Sansui's|dee...,NaN,NaN,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...
1,1,Waxman sells a variety of hardware products fo...,"{0: [""sells"", ""a"", ""variety"", ""of"", ""hardware""...",openai/gpt-oss-120b,sells|a|variety|of|hardware|products|for|the,Waxman,home repair market,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...
2,2,"Glaston , headquartered in Tampere , Finland ,...","{0: [""Glaston"", "","", ""headquartered"", ""in"", ""T...",openai/gpt-oss-120b,"Glaston|,|headquartered|in|Tampere|,|Finland|,...",NaN,NaN,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...
3,3,The overall weather trend will lead to a moder...,"{\n ""0"": [""The"", ""will"", ""lead"", ""to"", ""a"", ""...",openai/gpt-oss-120b,The|will|lead|to|a|by,NaN,overall weather trend,next Sunday,moderation of temperatures toward normal,synthetic-fpb-chronicle2050-yt-news-timebank-m...
4,4,The forecast indicates good predictability in ...,"{0: [""The"", ""forecast"", ""indicates"", ""in"", ""ba...",openai/gpt-oss-120b,The|forecast|indicates|in|based|on|guidance|fr...,GFS|ECMWF,mid to large-scale flow evolution,NaN,good predictability,synthetic-fpb-chronicle2050-yt-news-timebank-m...
5,5,StatesWest had proposed acquiring Mesa for $7 ...,"{0: [""had"", ""proposed"", ""acquiring"", ""for"", ""a...",openai/gpt-oss-120b,had|proposed|acquiring|for|and|one|share|of|a|...,StatesWest,Mesa,NaN,$7 a share|$3 a share,synthetic-fpb-chronicle2050-yt-news-timebank-m...
6,6,"But Kenneth Leon, a telecommunications analyst...","{0: [""But"", ""a"", ""telecommunications"", ""analys...",openai/gpt-oss-120b,But|a|telecommunications|analyst|with|Bear|Ste...,Kenneth Leon,BellSouth,five years,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...


In [3]:
col_names = df.iloc[: , 4:9]
col_names

,No Property,Source,Target,Date,Outcome
0,As|the|yen|surged|in|recent|years|Sansui's|dee...,NaN,NaN,NaN,NaN
1,sells|a|variety|of|hardware|products|for|the,Waxman,home repair market,NaN,NaN
2,"Glaston|,|headquartered|in|Tampere|,|Finland|,...",NaN,NaN,NaN,NaN
3,The|will|lead|to|a|by,NaN,overall weather trend,next Sunday,moderation of temperatures toward normal
4,The|forecast|indicates|in|based|on|guidance|fr...,GFS|ECMWF,mid to large-scale flow evolution,NaN,good predictability
5,had|proposed|acquiring|for|and|one|share|of|a|...,StatesWest,Mesa,NaN,$7 a share|$3 a share
6,But|a|telecommunications|analyst|with|Bear|Ste...,Kenneth Leon,BellSouth,five years,NaN


In [4]:
y_df = df.copy()
y_hat_df = df.copy()

### Need to Consider

> All cases below represent scenarios where semantic meaning is preserved but surface-level differences cause cosine similarity to underperform or fail entirely.

---

1. **Standardizing Dates**
   - **Dates appear in many formats:** `January 1st, 2037`, `2037-01-01`, `01/01/2037`
   - **Consider:** Normalize to a canonical format (e.g., ISO 8601) before embedding or comparison to prevent penalizing semantically identical dates with different representations.

2. **Word Order**
   - **same set, different order:** `2037, 2038` vs `2038, 2037`
   - **subset vs full set:** `2037, 2038` vs `2037`
   - **Consider:** Set-based matching (e.g., Jaccard similarity) for multi-value fields alongside cosine similarity.

3. **Word Tense / Morphology**
   - **plural vs singular:** `Long Bets` vs `Long Bet`
   -  **Consider:** Lemmatization before embedding (spaCy supports this) would normalize these cases and more relevant for Target and Outcome fields. now the affects it'll have on actual sentence.

4. **Missing or Extra Characters**
   - **unit dropped:** `2.0C or higher` vs `2.0 or higher`
   - **spacing artifact:** `U.S. Departmentof Energy` vs `U.S. Department of Energy`
   -  **Consider:** Lightweight string normalization (lowercase, strip punctuation, fix spacing) as a preprocessing step before embedding. Know the affects it'll have on actual sentence.

## Embed Sentences

In [5]:
property_results = []

for col_name in col_names:
    print(f"Embeddings for {col_name}")
    y_spacy_fe = SpacyFeatureExtraction(y_df, col_name)
    embed_y_df = y_spacy_fe.sentence_embeddings_extraction(attach_to_df=True)

    y_hat_spacy_fe = SpacyFeatureExtraction(y_hat_df, col_name)
    embed_y_hat_df = y_hat_spacy_fe.sentence_embeddings_extraction(attach_to_df=True)

    np.testing.assert_equal(len(embed_y_df), len(embed_y_hat_df))

    property_result = {
        'property_name': col_name,
        'y_data': embed_y_df,
        'y_hat_data': embed_y_hat_df
    }
    property_results.append(property_result)

Embeddings for No Property


100%|██████████| 7/7 [00:00<00:00, 522.63it/s]


Embeddings for Source


100%|██████████| 7/7 [00:00<00:00, 871.07it/s]


Embeddings for Target


100%|██████████| 7/7 [00:00<00:00, 686.34it/s]


Embeddings for Date


100%|██████████| 7/7 [00:00<00:00, 1313.59it/s]


Embeddings for Outcome


100%|██████████| 7/7 [00:00<00:00, 837.02it/s]


In [6]:
property_results[0]['y_data']

,Input_Index,Sentence,Raw Response,Model Name,No Property,Source,Target,Date,Outcome,Dataset Source,No Property Embedding,Source Embedding,Target Embedding,Date Embedding,Outcome Embedding
0,0,"As the yen soared in recent years, Sansui's de...","{0: [""As"", ""the"", ""yen"", ""surged"", ""in"", ""rece...",openai/gpt-oss-120b,As|the|yen|surged|in|recent|years|Sansui's|dee...,NaN,NaN,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",None,None,None,None
1,1,Waxman sells a variety of hardware products fo...,"{0: [""sells"", ""a"", ""variety"", ""of"", ""hardware""...",openai/gpt-oss-120b,sells|a|variety|of|hardware|products|for|the,Waxman,home repair market,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.5296, 0.57349, 0.22674, 0.47192, -0.77802,...","[0.022620002, 0.4353567, -0.13486333, 0.102556...",None,None
2,2,"Glaston , headquartered in Tampere , Finland ,...","{0: [""Glaston"", "","", ""headquartered"", ""in"", ""T...",openai/gpt-oss-120b,"Glaston|,|headquartered|in|Tampere|,|Finland|,...",NaN,NaN,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0060005, 0.103755, -0.06289, -0.296625, 0.0...",None,None,None,None
3,3,The overall weather trend will lead to a moder...,"{\n ""0"": [""The"", ""will"", ""lead"", ""to"", ""a"", ""...",openai/gpt-oss-120b,The|will|lead|to|a|by,NaN,overall weather trend,next Sunday,moderation of temperatures toward normal,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",None,"[-0.20438534, 0.37796497, 0.16734333, -0.04828...","[0.20051901, 0.26212, 0.0759155, -0.00803425, ...","[-0.19092801, 0.13687201, -0.0070824027, 0.107..."
4,4,The forecast indicates good predictability in ...,"{0: [""The"", ""forecast"", ""indicates"", ""in"", ""ba...",openai/gpt-oss-120b,The|forecast|indicates|in|based|on|guidance|fr...,GFS|ECMWF,mid to large-scale flow evolution,NaN,good predictability,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.10747714, 0.3004813, -0.07520829, 0.051865...",None,"[-0.40086502, 0.451215, -0.19719149, -0.119859..."
5,5,StatesWest had proposed acquiring Mesa for $7 ...,"{0: [""had"", ""proposed"", ""acquiring"", ""for"", ""a...",openai/gpt-oss-120b,had|proposed|acquiring|for|and|one|share|of|a|...,StatesWest,Mesa,NaN,$7 a share|$3 a share,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.49534, 0.045763, 0.4176, -0.28347, 0.51842,...",None,"[-0.21797733, 0.14856802, 0.106824994, 0.17588..."
6,6,"But Kenneth Leon, a telecommunications analyst...","{0: [""But"", ""a"", ""telecommunications"", ""analys...",openai/gpt-oss-120b,But|a|telecommunications|analyst|with|Bear|Ste...,Kenneth Leon,BellSouth,five years,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.25909, 0.17956, 0.28349, -0.336005, 0.1681...","[0.12799, 0.9218, 0.32255, -0.44427, 0.42114, ...","[0.097312, 0.107255846, 0.107334, -0.39513502,...",None


In [7]:
property_results[0]['y_hat_data']

,Input_Index,Sentence,Raw Response,Model Name,No Property,Source,Target,Date,Outcome,Dataset Source,No Property Embedding,Source Embedding,Target Embedding,Date Embedding,Outcome Embedding
0,0,"As the yen soared in recent years, Sansui's de...","{0: [""As"", ""the"", ""yen"", ""surged"", ""in"", ""rece...",openai/gpt-oss-120b,As|the|yen|surged|in|recent|years|Sansui's|dee...,NaN,NaN,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",None,None,None,None
1,1,Waxman sells a variety of hardware products fo...,"{0: [""sells"", ""a"", ""variety"", ""of"", ""hardware""...",openai/gpt-oss-120b,sells|a|variety|of|hardware|products|for|the,Waxman,home repair market,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.5296, 0.57349, 0.22674, 0.47192, -0.77802,...","[0.022620002, 0.4353567, -0.13486333, 0.102556...",None,None
2,2,"Glaston , headquartered in Tampere , Finland ,...","{0: [""Glaston"", "","", ""headquartered"", ""in"", ""T...",openai/gpt-oss-120b,"Glaston|,|headquartered|in|Tampere|,|Finland|,...",NaN,NaN,NaN,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0060005, 0.103755, -0.06289, -0.296625, 0.0...",None,None,None,None
3,3,The overall weather trend will lead to a moder...,"{\n ""0"": [""The"", ""will"", ""lead"", ""to"", ""a"", ""...",openai/gpt-oss-120b,The|will|lead|to|a|by,NaN,overall weather trend,next Sunday,moderation of temperatures toward normal,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",None,"[-0.20438534, 0.37796497, 0.16734333, -0.04828...","[0.20051901, 0.26212, 0.0759155, -0.00803425, ...","[-0.19092801, 0.13687201, -0.0070824027, 0.107..."
4,4,The forecast indicates good predictability in ...,"{0: [""The"", ""forecast"", ""indicates"", ""in"", ""ba...",openai/gpt-oss-120b,The|forecast|indicates|in|based|on|guidance|fr...,GFS|ECMWF,mid to large-scale flow evolution,NaN,good predictability,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.10747714, 0.3004813, -0.07520829, 0.051865...",None,"[-0.40086502, 0.451215, -0.19719149, -0.119859..."
5,5,StatesWest had proposed acquiring Mesa for $7 ...,"{0: [""had"", ""proposed"", ""acquiring"", ""for"", ""a...",openai/gpt-oss-120b,had|proposed|acquiring|for|and|one|share|of|a|...,StatesWest,Mesa,NaN,$7 a share|$3 a share,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.49534, 0.045763, 0.4176, -0.28347, 0.51842,...",None,"[-0.21797733, 0.14856802, 0.106824994, 0.17588..."
6,6,"But Kenneth Leon, a telecommunications analyst...","{0: [""But"", ""a"", ""telecommunications"", ""analys...",openai/gpt-oss-120b,But|a|telecommunications|analyst|with|Bear|Ste...,Kenneth Leon,BellSouth,five years,NaN,synthetic-fpb-chronicle2050-yt-news-timebank-m...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.25909, 0.17956, 0.28349, -0.336005, 0.1681...","[0.12799, 0.9218, 0.32255, -0.44427, 0.42114, ...","[0.097312, 0.107255846, 0.107334, -0.39513502,...",None


## Evaluation

![image.png](attachment:image.png)

### Map Words to Binary Classification Labels

Compares y and y_hat embeddings per property column and assigns TP, FN, FP, TN labels using cosine similarity.

In [8]:
def map_words_to_labels(y_df, y_hat_df, col_name):
    """
    Map word-level predictions to binary classification labels using cosine similarity.

    Compares ground truth to LLM predicted word for a given property column by computing
    cosine similarity between their embeddings. Each example is assigned a TP, FN, FP,
    or TN label based on the presence of embeddings and semantic similarity threshold.

    Parameters
    ----------
    y_df : pd.DataFrame
        Ground truth dataframe containing the property column and its embedding column.
    y_hat_df : pd.DataFrame
        Predicted dataframe containing the property column and its embedding column.
    col_name : str
        Name of the property column to evaluate (e.g., 'Source', 'Target', 'Date', 'Outcome').
        Expects a corresponding '{col_name} Embedding' column in both dataframes.

    Returns
    -------
    tps : list of dict
        True positives — y and y_hat are both present and cosine similarity >= 0.9.
    fns : list of dict
        False negatives — y is present but y_hat is absent or cosine similarity < 0.9.
    fps : list of dict
        False positives — y is absent but y_hat is present.
    tns : list of dict
        True negatives — both y and y_hat are absent.
    """

    len_dfs = len(y_df)
    
    tps = [] 
    fns = [] 
    fps = [] 
    tns = []

    for idx in range(len_dfs):

        y_word = y_df[f'{col_name}'][idx]
        y_embed = y_df[f'{col_name} Embedding'][idx]

        y_hat_word = y_hat_df[f'{col_name}'][idx]
        y_hat_embed = y_hat_df[f'{col_name} Embedding'][idx]

        # if idx < 7:
            # print(f"idx ({idx}): {y_word} vs {y_hat_word}")

        # y and y_hat are words and we need to see how similar
            # TP: same word 
            # FN: different words
        if y_embed is not None:
            if y_hat_embed is not None:
                # fallback for OOV/zero vectors
                if np.linalg.norm(y_embed) == 0 or np.linalg.norm(y_hat_embed) == 0:
                    cs = 1.0 if y_word == y_hat_word else 0.0
                else:
                    cs = EvaluationMetric.get_cosine_similarity(y_embed, y_hat_embed, per_row=False, idx=idx)
                if cs >= .9:
                    tp = {
                        'y_word': y_word,
                        'y_hat_word': y_hat_word,
                        'cs': cs,
                        'y': 1,
                        'y_hat': 1
                    }
                    tps.append(tp)
                    # print(f"idx ({idx}): {y_word} vs {y_hat_word} => tp")
                else:
                    fn = {
                        'y_word': y_word,
                        'y_hat_word': y_hat_word,
                        'cs': cs,
                        'y': 1,
                        'y_hat': 0
                    }
                    fns.append(fn)
                    # print(f"idx ({idx}): {y_word} vs {y_hat_word} => fn")
            # y_hat is None
            else:
                fn = {
                    'y_word': y_word,
                    'y_hat_word': y_hat_word,
                    'cs': None,
                    'y': 1,
                    'y_hat': 0
                }
                fns.append(fn)
                # print(f"idx ({idx}): {y_word} vs {y_hat_word} => fn")


        # y is NOT a word and y_hat is either a word or not a word
        elif y_embed is None:

            # FP: hallucinate a word
            if y_hat_embed is not None:
                fp = {
                    'y_word': y_word,
                    'y_hat_word': y_hat_word,
                    'cs': None,
                    'y': 0,
                    'y_hat': 1
                }
                fps.append(fp)
                # print(f"idx ({idx}): {y_word} vs {y_hat_word} => fp")

            # y_hat is None
            # TN: correcly predict nothing
            else:
                tn = {
                    'y_word': y_word,
                    'y_hat_word': y_hat_word,
                    'cs': None,
                    'y': 0,
                    'y_hat': 0
                }
                tns.append(tn)
                # print(f"idx ({idx}): {y_word} vs {y_hat_word} => tn")

    return tps, fns, fps, tns


### Compute Metrics Per Property

Builds classification report and confusion matrix across Source, Target, Date, and Outcome.

In [9]:
seed = 3
model_name = 'openai/gpt-oss-120b'
eval_report_dfs = []
metrics_summary = []

for property_results_idx in range(len(property_results)):
    property_result = property_results[property_results_idx]

    property_name = property_result['property_name']
    print(f"Classification Results from: {property_name}")
    
    y_df = property_result['y_data']
    y_hat_df = property_result['y_hat_data']
    
    tps, fns, fps, tns = map_words_to_labels(y_df, y_hat_df, property_name)
    print(f"\t#TP: {len(tps)}, \n\t#FN: {len(fns)}, \n\t#FP: {len(fps)}, \n\t#TN: {len(tns)}")

    # for fn in fns:
    #     print(f"\n\tFN: {fn}")

    print(f"\tConvert to DFs")
    tps_df = pd.DataFrame(tps)
    fns_df = pd.DataFrame(fns)
    fps_df = pd.DataFrame(fps)
    tns_df = pd.DataFrame(tns)
    
    print(f"\tConcat DFs")
    eval_report_df = DataProcessing.concat_dfs([tps_df, fns_df, fps_df, tns_df])
    # eval_report_dfs.append(eval_report_df)
    actual_labels = eval_report_df['y']
    llm_predicted_labels = eval_report_df['y_hat']

    print(f"\tClassification Report")
    # Classification report: precision, recall, f1, test accuracy
    eval_report = EvaluationMetric.eval_classification_report(actual_labels, llm_predicted_labels)
    # eval_reports[f"{label_name}-{model_name}"] = eval_report
    
    # Confusion matrix
    confusion_mat, tn, fp, fn, tp = EvaluationMetric.get_confusion_matrix(actual_labels, llm_predicted_labels, by_category=True)
    print(f"Confusion Matrix:\n{confusion_mat}\n")
    # Save confusion matrix visualization
    # DataVisualizing.confusion_matrix(
    #     model_name,
    #     confusion_mat, 
    #     save_path, 
    #     include_version=False
    # )
    # print(f"✓ Saved confusion matrix: confusion_matrix_{model_name}.png\n")
            
    # Build unified metrics row
    metrics_row = {
        'seed': seed,
        'model': model_name,
        'property': property_name,
        # Get classification report metrics
        'test_accuracy': eval_report.get('accuracy', None),
        'precision_class_0': eval_report.get('0', {}).get('precision', None),
        'precision_class_1': eval_report.get('1', {}).get('precision', None),
        'recall_class_0': eval_report.get('0', {}).get('recall', None),
        'recall_class_1': eval_report.get('1', {}).get('recall', None),
        'f1_class_0': eval_report.get('0', {}).get('f1-score', None),
        'f1_class_1': eval_report.get('1', {}).get('f1-score', None),
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp
    }
    # precision recall
    # roc
    metrics_summary.append(metrics_row)

# Save unified metrics summary
metrics_summary_df = pd.DataFrame(metrics_summary)

Classification Results from: No Property
	#TP: 7, 
	#FN: 0, 
	#FP: 0, 
	#TN: 0
	Convert to DFs
	Concat DFs
	Classification Report
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         7

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7

Confusion Matrix:
[[0 0]
 [0 7]]

Classification Results from: Source
	#TP: 4, 
	#FN: 0, 
	#FP: 0, 
	#TN: 3
	Convert to DFs
	Concat DFs
	Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         4

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7

Confusion Matrix:
[[3 0]
 [0 4]]

Classification Results from: Target
	#TP: 5, 
	#FN: 0, 
	#FP: 0, 
	#TN: 2
	Co

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data_processing.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, axis=axis, ignore_index=ignore_index)


In [10]:
metrics_summary_df

,seed,model,property,test_accuracy,precision_class_0,precision_class_1,recall_class_0,recall_class_1,f1_class_0,f1_class_1,tn,fp,fn,tp
0,3,openai/gpt-oss-120b,No Property,1.0,NaN,1.0,NaN,1.0,NaN,1.0,0,0,0,7
1,3,openai/gpt-oss-120b,Source,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3,0,0,4
2,3,openai/gpt-oss-120b,Target,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2,0,0,5
3,3,openai/gpt-oss-120b,Date,1.0,1.0,1.0,1.0,1.0,1.0,1.0,5,0,0,2
4,3,openai/gpt-oss-120b,Outcome,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4,0,0,3


In [11]:
eval_save_path = os.path.join(base_data_path, 'classification_results', 'properties')

DataProcessing.save_to_file(
    metrics_summary_df, 
    path=eval_save_path, 
    prefix='metrics_summary_llms', 
    save_file_type='csv', 
    include_version=True,
    )
print(f"✓ Saved metrics summary to: {os.path.join(eval_save_path, 'metrics_summary_llms.csv')}")

Using file number: 4
Saving CSV file to: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data/classification_results/properties/metrics_summary_llms-v4.csv
✓ Saved metrics summary to: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/properties_extraction_experiments/../data/classification_results/properties/metrics_summary_llms.csv
